# 05 · Analysis

This notebook turns the outputs of the fine-tuned models (notebook 04) into the results of the paper:
decision thresholds, decision changes between repeated readings, discrimination (AUROC), direction agreement,
and the paired bootstrap that gives every difference from the baseline its 95% confidence interval.

| | |
|---|---|
| Reads | `EVAL_RECORDS` (one row per evaluation recording), `LOGITS/{tag}.npz` (outputs of every fine-tuned model for every evaluation recording) and `MC_DROPOUT/{tag}.npz` (probabilities with Monte Carlo dropout), all written by notebooks 02 and 04 |
| Writes | three tables in `SUMMARY`. They are documented in the last section, **Files written**. The table notebook (07) and the figure notebook (08) read these tables and nothing else from this notebook |
| Needs | no GPU. About 5 min and less than 2 GB of memory on a desktop computer |

## Outline

1. **Evaluation records**: which rows are the validation fold, the test sets and the pairs.
2. **Probabilities and readings**: from logits to probabilities; single read, window average, mixed reading.
3. **Decision thresholds**: 90% sensitivity in the PTB-XL validation fold, for each class and each predictor, then fixed.
4. **Outcomes**: decision change between adjacent segments and between recordings, individual decisions, AUROC, direction agreement.
5. **Paired bootstrap**: resamples of recordings or pairs, the same for every configuration.
6. **Configurations**: what is evaluated (three foundation models; baseline, the three remedies and their parts; alternatives; ensemble; Monte Carlo dropout).
7. **Summary tables** and the direct comparison of the three remedies with the ensemble and with Monte Carlo dropout.
8. **Files written**.

## Words used in this notebook

- **Segment**: one of the two adjacent, non-overlapping 5-s halves of a 10-s recording. A model reads one segment as its ECG.
- **Predictor**: what turns a segment into five probabilities: one fine-tuned model (one seed), one head and one way of reading.
- **Configuration**: the same training, head and reading for the three seeds, for example `student`, `pen300`, `mixed0.9` (the three remedies).
  Every outcome is calculated for the predictor of each seed and then averaged over the seeds. The outputs of the models of different seeds are never averaged (except in the ensemble, where this is the method itself).
- **Unit**: what is resampled in the bootstrap, a recording or a pair of recordings.

In [ ]:
%run ./00_setup_and_settings.ipynb

In [ ]:
import time

import numpy as np
import pandas as pd

T_START = time.time()
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 100)

## Evaluation records

`EVAL_RECORDS` has one row per evaluation recording, and every logits file has its rows in the same order. The column `set` says what a row is used for:

| Set | Used for | Unit |
|---|---|---|
| `ptbxl_val` | decision thresholds (and the choice of the dropout rate) | – |
| `ptbxl_test` | AUROC (all rows); decision change between adjacent segments (rows with `adjacent_ok` = 1) | recording |
| `mimic_test`, `code15_test`, `echonext_test` | decision change between adjacent segments, AUROC | recording |
| `mimic_pairs_same_report`, `mimic_pairs_all` | decision change between recordings (roles `a` and `b` of a pair) | pair |
| `code15_cd_change`, `code15_normal_change` | direction agreement (role `pos` = ECG with the label, `neg` = ECG without it) | pair |

In PTB-XL the comparison of adjacent segments uses a subset of the test fold (recordings without statements of premature beats or pacing, `adjacent_ok` = 1),
whereas the AUROC uses the whole test fold. The other test sets were built with these exclusions, so all their rows are used for both.

In [ ]:
REC = pd.read_csv(EVAL_RECORDS)
SET = REC["set"].values.astype(str)
N_REC = len(REC)
assert set(SET) <= set(EVAL_SETS), "unknown set name in EVAL_RECORDS"
Y = REC[CLASSES].values.astype(np.float32)                        # reference labels (rows, 5 classes); NaN where a dataset has no label for a class

DATASETS = ["ptbxl", "mimic", "code15", "echonext"]
VAL = np.flatnonzero(SET == "ptbxl_val")                           # rows of the validation fold
TEST = {d: np.flatnonzero(SET == f"{d}_test") for d in DATASETS}   # rows of each test set (AUROC)
ADJACENT_ROWS = {d: r[REC["adjacent_ok"].values[r] == 1] for d, r in TEST.items()}   # test rows used for the comparison of adjacent segments


def pair_rows(set_name, first, second):
    """Rows of the two recordings of every pair of a set: two arrays in the same order of pairs."""
    q = REC[REC["set"] == set_name]
    a = q[q["role"] == first].sort_values("pair_id"); b = q[q["role"] == second].sort_values("pair_id")
    assert len(a) == len(b) and (a["pair_id"].values == b["pair_id"].values).all() and (a["patient"].values == b["patient"].values).all()
    return a.index.values, b.index.values


PAIRS = {"between_same_report": pair_rows("mimic_pairs_same_report", "a", "b"),      # pairs with identical machine reports: the main comparison between recordings
         "between_all": pair_rows("mimic_pairs_all", "a", "b"),
         "direction_cd": pair_rows("code15_cd_change", "pos", "neg"),                # conduction label present in one ECG of the pair
         "direction_normal": pair_rows("code15_normal_change", "pos", "neg")}        # normal label present in one ECG of the pair

assert not np.isnan(Y[np.r_[VAL, TEST["ptbxl"], TEST["mimic"]]]).any()               # five labels in PTB-XL and MIMIC-IV-ECG
assert not np.isnan(Y[TEST["code15"]][:, [CLASSES.index("CD"), CLASSES.index("NORM")]]).any() and not np.isnan(Y[TEST["echonext"], CLASSES.index("HYP")]).any()
print(f"{N_REC:,} evaluation recordings")
print("validation fold:", len(VAL), "| test sets:", {d: len(r) for d, r in TEST.items()})
print("recordings for adjacent segments:", {d: len(r) for d, r in ADJACENT_ROWS.items()}, "| pairs:", {k: len(a) for k, (a, b) in PAIRS.items()})

## Probabilities and readings

### From logits to probabilities

The five diagnostic classes are not exclusive, so every class has its own output. A logits file holds the logit $z$ of each class, and the probability is the logistic function $p = 1 / (1 + e^{-z})$.

A logits file contains, for each head, two arrays (first axis = rows of `EVAL_RECORDS`, second axis = the two segments of the recording, last axis = the five classes):

- `{head}_single` (rows, 2, 5): one read of the whole 5-s segment,
- `{head}_windows` (rows, 2, 25, 5): the 25 overlapping windows of 2.5 s of the segment.

`head` is `model` (the output layer that was trained with the network) or `pen{g}` (the linear head refitted with penalty weight g; `pen0` is refitted without a penalty).

### Readings

A reading turns these outputs into one probability for each class of a segment.

| Name | Probability used for the decision |
|---|---|
| `single` | probability from the one read of the segment |
| `windows` | window average: mean of the probabilities of the 25 windows (probabilities are averaged, not logits) |
| `mixed0.9` | mixed reading of the final setting: `MIX_SINGLE` × single + (1 − `MIX_SINGLE`) × window average = 0.1 × single + 0.9 × window average |
| `mixed0.25`, `mixed0.5`, `mixed0.75` | the same with a weight of 0.25, 0.5 or 0.75 for the window average (Supplementary Fig. 2b) |

The number in `mixed{w}` is the weight of the **window average**, as in the paper. A weight of 0 is the single read and a weight of 1 is the window average alone.
All windows lie within the segment, so every reading uses the same 5 s of signal.

Probabilities are calculated in double precision and then kept in single precision, which halves the memory.
Single precision resolves differences of about 6 × 10⁻⁸ close to 1. Two probabilities that differ by less are equal after this step and count as a tie in direction agreement and in the AUROC.

In [ ]:
MIXED = f"mixed{1 - MIX_SINGLE:g}"                                 # name of the mixed reading of the final setting: "mixed0.9"
# weight of the single read in each reading (the weight of the window average is one minus this value)
SINGLE_WEIGHT = {"single": 1.0, "mixed0.25": 0.75, "mixed0.5": 0.5, "mixed0.75": 0.25, MIXED: MIX_SINGLE, "windows": 0.0}


def sigmoid(z):
    """Logistic function, in double precision."""
    return 1.0 / (1.0 + np.exp(-z.astype(np.float64)))


def load_logits(tag):
    """Open the logits file of a fine-tuned model and check that its rows are the rows of EVAL_RECORDS."""
    z = np.load(LOGITS / f"{tag}.npz")
    assert (z["record"] == REC["record"].values).all() and (z["set"] == SET).all(), f"{tag}: the rows are not in the order of EVAL_RECORDS"
    return z


def read_probabilities(z, head, readings):
    """Probabilities (rows, 2 segments, 5 classes) of the requested readings for one head of one model."""
    p_single = sigmoid(z[f"{head}_single"])
    p_windows = sigmoid(z[f"{head}_windows"]).mean(2) if any(r != "single" for r in readings) else None      # mean probability over the 25 windows
    out = {}
    for r in readings:
        a = SINGLE_WEIGHT[r]
        p = p_single if a == 1 else p_windows if a == 0 else a * p_single + (1 - a) * p_windows
        out[r] = p.astype(np.float32)
    return out

## Decision thresholds

A model gives a probability, and a decision needs a threshold. For each class the threshold is set at 90% sensitivity (`SENSITIVITY`) in the PTB-XL validation fold:

- the two segments of every validation recording are pooled (2 × 2,146 segments), each with the labels of its recording;
- the probabilities of the positive segments are sorted, and the threshold is the value below which a fraction 1 − `SENSITIVITY` of them lies,
  so that at least 90% of the positive segments have a probability at or above the threshold;
- a decision is positive when the probability is at or above the threshold.

Thresholds are set **separately for every predictor**, that is for every model (seed), head and reading, because a change of head or of reading shifts the distribution of the probabilities.
Each predictor therefore works at the same sensitivity in the validation fold, and configurations are compared at equal sensitivity rather than at equal threshold.
Once set, the five thresholds of a predictor are **fixed**: they are used unchanged in every evaluation set and in every bootstrap resample.

In [ ]:
def thresholds(p):
    """Five thresholds of one predictor from the validation fold. p: probabilities (rows, 2, 5)."""
    y = np.concatenate([Y[VAL], Y[VAL]]); q = np.concatenate([p[VAL, 0], p[VAL, 1]])       # both segments pooled
    thr = np.empty(len(CLASSES), np.float32)
    for j in range(len(CLASSES)):
        pos = np.sort(q[y[:, j] == 1, j])                                                  # probabilities of the positive segments, ascending
        thr[j] = pos[min(int(np.floor((1 - SENSITIVITY) * len(pos))), len(pos) - 1)]      # the lowest 10% lie below the threshold
    return thr

## Outcomes

Every outcome is first calculated **for each unit** (recording or pair), and the reported value is the mean over the units. Keeping the values of the units is what makes the bootstrap simple:
the value in a resample is the weighted mean of the same numbers.

### Decision change between adjacent segments (primary outcome)

For a recording, the five decisions for the first segment are compared with the five decisions for the second segment. The recording counts as changed when the decision for **at least one** of the five classes differs.
The outcome is the proportion of recordings with a change (%), in the PTB-XL test recordings with `adjacent_ok` = 1 and in the test sets of MIMIC-IV-ECG, CODE-15% and EchoNext.

### Decision change between recordings

The same definition for two recordings of the same patient taken 1 min to 24 h apart: the decisions for the **first segment** of the first recording are compared with those for the first segment of the second recording,
so that both kinds of comparison use inputs of the same length. The outcome is the proportion of pairs with a change (%), for pairs with identical machine reports (no documented change; the main comparison between recordings) and for all pairs.

### Individual decisions and diagnostic classes

- **Individual decisions changed**: for a unit, the proportion of its five decisions that differ between the two readings; the outcome is the mean over units (%), which equals the mean over the five classes of the proportions for each class.
- **By diagnostic class**: the proportion of recordings in which the decision for that class differs between the adjacent segments (PTB-XL).

In [ ]:
def decisions_changed(D, rows_1, segment_1, rows_2, segment_2):
    """(units, 5) array: True where the decision for a class differs between two readings of a unit. D: decisions (rows, 2, 5)."""
    return D[rows_1, segment_1] != D[rows_2, segment_2]

### AUROC

Discrimination is measured by the area under the receiver operating characteristic curve (AUROC), which uses the probabilities and no threshold:

| Outcome | Rows | Score | Reference |
|---|---|---|---|
| `auroc_ptbxl_{CLASS}`, `auroc_ptbxl` | PTB-XL test fold | probability of each class; mean of the five AUROCs | statements assigned by cardiologists |
| `auroc_mimic_{CLASS}`, `auroc_mimic` | MIMIC-IV-ECG test set | probability of each class; mean of the five AUROCs | machine statements (approximate) |
| `auroc_code15_cd`, `auroc_code15_normal` | CODE-15% test set | probability of conduction disturbance; of normal ECG | labels from physician reports |
| `auroc_echonext` | EchoNext test set | probability of hypertrophy | left ventricular wall thickness of 13 mm or more on echocardiography |

Each AUROC is calculated for the first segments and for the second segments of the recordings and the two values are averaged, because a model reads one segment as its ECG.

The function below gives the AUROC in every bootstrap resample at once (the bootstrap itself is described in the next section).
A resample is described by a weight for each recording, the number of times it was drawn, and the AUROC with weights is

$$\mathrm{AUROC} = \frac{\sum_{i \in \text{positive}} w_i \, \big(W^{-}_{<}(i) + \tfrac{1}{2} W^{-}_{=}(i)\big)}{\sum_{i \in \text{positive}} w_i \; \sum_{k \in \text{negative}} w_k},$$

where $W^{-}_{<}(i)$ and $W^{-}_{=}(i)$ are the total weights of the negative recordings with a lower score than recording $i$ and with the same score (ties count one half).
Both come from one cumulative sum over the negative recordings sorted by score, so no loop over resamples is needed. With all weights equal to one this is the ordinary AUROC.

In [ ]:
def auroc_resampled(score, label, weights):
    """AUROC in every resample. score (n,), label (n,) of 0 and 1, weights (resamples, n) -> (resamples,)."""
    pos = label == 1
    s_neg = score[~pos]; order = np.argsort(s_neg, kind="stable"); s_neg = s_neg[order]
    cum = np.zeros((weights.shape[0], len(s_neg) + 1), np.float32)
    np.cumsum(weights[:, ~pos][:, order], axis=1, out=cum[:, 1:])          # cum[:, k] = weight of the k negatives with the lowest scores (whole numbers: exact in single precision)
    lo = np.searchsorted(s_neg, score[pos], "left")                        # number of negatives with a lower score than each positive
    hi = np.searchsorted(s_neg, score[pos], "right")                       # ... with a lower or equal score
    w_pos = weights[:, pos]
    numerator = 0.5 * (w_pos * (cum[:, lo] + cum[:, hi])).sum(1, dtype=np.float64)     # lower + half of the ties = (lower + lower-or-equal) / 2
    return numerator / (w_pos.sum(1, dtype=np.float64) * cum[:, -1])


def auroc(score, label):
    """Ordinary AUROC (every unit once)."""
    return float(auroc_resampled(score, label, np.ones((1, len(score)), np.float32))[0])


# the AUROC outcomes of single classes: (name, dataset, index of the class)
AUROC_TARGETS = [(f"auroc_{d}_{c}", d, j) for d in ("ptbxl", "mimic") for j, c in enumerate(CLASSES)]
AUROC_TARGETS += [("auroc_code15_cd", "code15", CLASSES.index("CD")), ("auroc_code15_normal", "code15", CLASSES.index("NORM")), ("auroc_echonext", "echonext", CLASSES.index("HYP"))]

### Direction agreement

Stability must not be gained by ignoring real change. In CODE-15% there are pairs of ECGs of the same patient in which a label (conduction disturbance, or normal ECG) is present in one ECG only.
For such a pair the corresponding output should be higher in the ECG with the label. **Direction agreement** is the proportion of pairs in which this is the case (%).
It uses the probabilities of the first segments and no threshold, and a pair with two equal probabilities counts one half.

In [ ]:
def direction_agreement(p, with_label, without_label, j):
    """Per pair: 1 if the probability of class j (first segment) is higher in the ECG with the label, 0.5 if both are equal, 0 otherwise."""
    q1, q0 = p[with_label, 0, j], p[without_label, 0, j]
    return (q1 > q0) + 0.5 * (q1 == q0)

### Names of the outcomes

The names below are used as keys in this notebook and as the first part of the column names in the summary tables.

In [ ]:
OUT_ADJACENT = [f"adjacent_{d}" for d in DATASETS]                                 # decision change between adjacent segments (any of the five classes), % of recordings
OUT_BETWEEN = ["between_same_report", "between_all"]                               # decision change between recordings, % of pairs
OUT_INDIVIDUAL = [f"individual_{o}" for o in OUT_ADJACENT + OUT_BETWEEN]           # individual decisions changed, %
OUT_BY_CLASS = [f"adjacent_ptbxl_{c}" for c in CLASSES]                            # decision change between adjacent segments for one class, PTB-XL, % of recordings
OUT_DIRECTION = ["direction_cd", "direction_normal"]                               # direction agreement in CODE-15%, % of pairs
OUT_AUROC = ["auroc_ptbxl"] + [f"auroc_ptbxl_{c}" for c in CLASSES] + ["auroc_mimic"] + [f"auroc_mimic_{c}" for c in CLASSES] + ["auroc_code15_cd", "auroc_code15_normal", "auroc_echonext"]
OUT_PERCENT = OUT_ADJACENT + OUT_BETWEEN + OUT_INDIVIDUAL + OUT_BY_CLASS + OUT_DIRECTION     # outcomes that are means over units, in percent
OUTCOMES = OUT_PERCENT + OUT_AUROC

# the set of units that is resampled for each outcome
UNIT_SET = {}
for _d in DATASETS:
    UNIT_SET[f"adjacent_{_d}"] = UNIT_SET[f"individual_adjacent_{_d}"] = "ptbxl_test_adjacent" if _d == "ptbxl" else f"{_d}_test"
for _o in OUT_BY_CLASS:
    UNIT_SET[_o] = "ptbxl_test_adjacent"
UNIT_SET.update({"between_same_report": "mimic_pairs_same_report", "individual_between_same_report": "mimic_pairs_same_report", "between_all": "mimic_pairs_all", "individual_between_all": "mimic_pairs_all",
                 "direction_cd": "code15_cd_change", "direction_normal": "code15_normal_change"})
for _o in OUT_AUROC:
    UNIT_SET[_o] = _o.split("_")[1] + "_test"
N_UNITS = {"ptbxl_test": len(TEST["ptbxl"]), "ptbxl_test_adjacent": len(ADJACENT_ROWS["ptbxl"]), "mimic_test": len(TEST["mimic"]), "code15_test": len(TEST["code15"]), "echonext_test": len(TEST["echonext"]),
           "mimic_pairs_same_report": len(PAIRS["between_same_report"][0]), "mimic_pairs_all": len(PAIRS["between_all"][0]),
           "code15_cd_change": len(PAIRS["direction_cd"][0]), "code15_normal_change": len(PAIRS["direction_normal"][0])}
display(pd.DataFrame({"outcome": OUTCOMES, "resampled set": [UNIT_SET[o] for o in OUTCOMES], "units": [N_UNITS[UNIT_SET[o]] for o in OUTCOMES]}))

## Paired bootstrap

Confidence intervals come from a bootstrap over **units**: recordings for the outcomes of a test set, pairs for the outcomes of a set of pairs. A resample draws as many units as the set has, with replacement.

- **Thresholds are fixed.** They come from the validation fold, which is not resampled, so a resample changes only which test units are counted.
- **The same resamples are used for every configuration and every seed.** A difference between two configurations is calculated within each resample, on the same units, so the variation between units that the two share cancels (paired comparison).
- **Models of the same seed are paired**, and the difference is averaged over the three seeds within each resample. The reported value is the mean difference over the three seeds,
  and the 95% confidence interval runs from the 2.5th to the 97.5th percentile of the resampled differences.
- **Number of resamples**: `N_BOOT` (2,000) for decision change and direction agreement, `N_BOOT_AUROC` (1,000) for the AUROC, which costs more to recalculate.
- A difference is called **significant** when its interval excludes zero. No adjustment is made for multiple comparisons.
- **Relative reductions** (difference as a percentage of the baseline value) are calculated from the means of the three seeds and have no interval.

The same resamples also give the confidence interval of a value itself (Fig. 2a,b and Supplementary Table 3): the value of each unit is averaged over the three seeds, and the percentiles of its resampled means are taken.

### Resamples

Every set of units has its own random generator, seeded with `BOOT_SEED` and the position of the set in the list `BOOT_SETS`, so the resamples of one set do not depend on any other set.
A resample is stored as a row of weights: the number of times each unit was drawn. Each set gets `N_BOOT` resamples, and the AUROC uses the first `N_BOOT_AUROC` of them.
The PTB-XL test recordings used for adjacent segments are resampled as a set of their own, so that every resample has as many recordings as that subset.

In [ ]:
def bootstrap_weights(n_units, n_boot, rng):
    """(n_boot, n_units) array: the number of times each unit is drawn in each resample of n_units units with replacement."""
    drawn = rng.integers(0, n_units, (n_boot, n_units))
    return np.stack([np.bincount(i, minlength=n_units) for i in drawn]).astype(np.float32)


BOOT_SETS = list(N_UNITS)                                          # the order of this list fixes the generator of each set
W = {name: bootstrap_weights(N_UNITS[name], N_BOOT, np.random.default_rng([BOOT_SEED, k])) for k, name in enumerate(BOOT_SETS)}
for _k, _name in enumerate(BOOT_SETS):
    print(f"{_k}  {_name:24s} {W[_name].shape[1]:6d} units, {W[_name].shape[0]} resamples")

### All outcomes of one predictor

`evaluate` applies the definitions above to the probabilities of one predictor. It returns the thresholds, the value of every outcome for every unit (for the bootstrap of the proportions),
and each AUROC with its value in every resample.

In [ ]:
def evaluate(p, ptbxl_only=False):
    """All outcomes of one predictor. p: probabilities (rows of EVAL_RECORDS, 2 segments, 5 classes).
    ptbxl_only: the predictor has outputs for the PTB-XL rows only (Monte Carlo dropout), so only the outcomes of PTB-XL are calculated."""
    thr = thresholds(p)
    D = p >= thr                                                   # decisions (rows, 2, 5) with the fixed thresholds
    unit, point, boot = {}, {}, {}
    for d in (["ptbxl"] if ptbxl_only else DATASETS):              # adjacent segments: first against second segment of the same recording
        r = ADJACENT_ROWS[d]; ch = decisions_changed(D, r, 0, r, 1)
        unit[f"adjacent_{d}"] = ch.any(1); unit[f"individual_adjacent_{d}"] = ch.mean(1)
        if d == "ptbxl":
            for j, c in enumerate(CLASSES):
                unit[f"adjacent_ptbxl_{c}"] = ch[:, j]
    if not ptbxl_only:
        for name in OUT_BETWEEN:                                   # between recordings: first segment of one recording against first segment of the other
            a, b = PAIRS[name]; ch = decisions_changed(D, a, 0, b, 0)
            unit[name] = ch.any(1); unit[f"individual_{name}"] = ch.mean(1)
        unit["direction_cd"] = direction_agreement(p, *PAIRS["direction_cd"], CLASSES.index("CD"))
        unit["direction_normal"] = direction_agreement(p, *PAIRS["direction_normal"], CLASSES.index("NORM"))
    for name, d, j in AUROC_TARGETS:                               # AUROC: mean of the first and the second segments, as a value and in every resample
        if ptbxl_only and d != "ptbxl":
            continue
        r = TEST[d]; y = Y[r, j]; w = W[f"{d}_test"][:N_BOOT_AUROC]
        point[name] = float(np.mean([auroc(p[r, h, j], y) for h in (0, 1)]))
        boot[name] = np.mean([auroc_resampled(p[r, h, j], y, w) for h in (0, 1)], 0)
    for d in ("ptbxl", "mimic"):                                   # mean of the five classes
        if f"auroc_{d}_{CLASSES[0]}" in point:
            point[f"auroc_{d}"] = float(np.mean([point[f"auroc_{d}_{c}"] for c in CLASSES])); boot[f"auroc_{d}"] = np.mean([boot[f"auroc_{d}_{c}"] for c in CLASSES], 0)
    val_auroc = float(np.mean([auroc(p[VAL, h, j], Y[VAL, j]) for h in (0, 1) for j in range(len(CLASSES))]))      # validation fold, mean of five classes and two segments
    return {"thresholds": thr, "val_auroc": val_auroc, "unit": {k: v.astype(np.float32) for k, v in unit.items()}, "auroc": point, "auroc_boot": boot}

A check of the two building blocks on one predictor (ECGFounder, baseline, seed 0, single read): the thresholds give the intended sensitivity in the validation fold,
and `auroc` and `auroc_resampled` agree with scikit-learn (the latter with the weights of one resample passed as `sample_weight`).

In [ ]:
from sklearn.metrics import roc_auc_score

_p = read_probabilities(load_logits(model_tag("ecgfounder", "baseline", 0)), "model", ["single"])["single"]
_thr = thresholds(_p)
_positive = np.concatenate([Y[VAL], Y[VAL]]) == 1                                         # positive validation segments (segments, 5)
_called = np.concatenate([_p[VAL, 0], _p[VAL, 1]]) >= _thr                                # decisions for the validation segments
_sens = [float(_called[_positive[:, j], j].mean()) for j in range(len(CLASSES))]
print("thresholds:", {c: round(float(t), 3) for c, t in zip(CLASSES, _thr)})
print("sensitivity in the validation fold:", dict(zip(CLASSES, np.round(_sens, 4).tolist())))
_r = TEST["ptbxl"]; _j = CLASSES.index("MI"); _w = W["ptbxl_test"][:3]
print("AUROC, myocardial infarction, first segments:", auroc(_p[_r, 0, _j], Y[_r, _j]), "| scikit-learn:", roc_auc_score(Y[_r, _j], _p[_r, 0, _j]))
print("in three resamples:", auroc_resampled(_p[_r, 0, _j], Y[_r, _j], _w), "| scikit-learn:", [roc_auc_score(Y[_r, _j], _p[_r, 0, _j], sample_weight=w) for w in _w])
assert all(s >= SENSITIVITY for s in _sens) and abs(auroc(_p[_r, 0, _j], Y[_r, _j]) - roc_auc_score(Y[_r, _j], _p[_r, 0, _j])) < 1e-12

### Values, paired differences and confidence intervals

`bootstrap_summary` gives, for one outcome, the mean over the seeds with its percentile interval: of the value of a configuration, or of its paired difference from a reference configuration.
Proportions are returned in percent, so their differences are in percentage points.

In [ ]:
def bootstrap_summary(cfg, name, ref=None):
    """Point estimate, lower and upper limit of the 95% CI of an outcome of a configuration (ref=None) or of its paired difference from the configuration `ref`.
    cfg and ref map seed -> output of evaluate(). Models of the same seed are paired, and the mean over the seeds is taken within every resample."""
    paired = ref is not None
    seeds = [s for s in cfg if not paired or s in ref]
    if name in OUT_AUROC:
        point = np.mean([cfg[s]["auroc"][name] - (ref[s]["auroc"][name] if paired else 0.0) for s in seeds])
        boot = np.mean([cfg[s]["auroc_boot"][name] - (ref[s]["auroc_boot"][name] if paired else 0.0) for s in seeds], 0)      # AUROC (or its difference) in every resample, averaged over the seeds
    else:
        v = np.mean([cfg[s]["unit"][name] - (ref[s]["unit"][name] if paired else 0.0) for s in seeds], 0).astype(np.float32)  # value (or difference) of each unit, averaged over the seeds
        w = W[UNIT_SET[name]]
        point = 100 * v.mean(dtype=np.float64)
        boot = 100 * (w @ v) / w.shape[1]                          # mean over the units drawn in each resample
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return float(point), float(lo), float(hi)

## Configurations

A configuration is a combination of **training** (kind of fine-tuned model, as named in notebook 03), **head** and **reading**.

| Training | Meaning |
|---|---|
| `baseline` | fine-tuned on labels only, the whole segment as input |
| `student` | distillation from a teacher trained on 2.5-s windows (the training remedy) |
| `windows`, `agreement`, `selfdistill` | alternatives to distillation, ECGFounder only: training on 2.5-s windows, direct agreement loss, self-distillation |
| `ensemble10` | mean probability of ten baseline models (ECGFounder) |
| `dropout10` … `dropout50` | baseline trained with dropout at a rate of 0.1 … 0.5 (ECGFounder) |

The **baseline** of the paper is `baseline`, `model`, `single`. The **three remedies** (final setting) are `student`, `pen300`, `mixed0.9`. Every configuration is compared with the baseline of the same foundation model.

For each of the three foundation models:

| Purpose | Training | Head | Reading |
|---|---|---|---|
| each remedy alone, in part and together (Fig. 2, Table 2, Supplementary Table 4) | `baseline`, `student` | `model`, `pen300` | `single`, `mixed0.9` |
| penalty weight (Fig. 5, Supplementary Fig. 2c, Supplementary Table 4) | `student` | `pen0`, `pen30`, `pen100`, `pen1000` | `mixed0.9` |
| weight of the window average (Supplementary Fig. 2b) | `student` | `pen300`, `model` | `mixed0.25`, `mixed0.5`, `mixed0.75`, `windows` |

and for ECGFounder also the alternative training methods (model head, single read; Supplementary Fig. 2a), the ensemble and Monte Carlo dropout (Fig. 3), which are evaluated in the sections that follow.

The cell below lists what is evaluated from the logits files. A file or a head that is missing is reported and left out.

In [ ]:
HEAD = f"pen{PENALTY_WEIGHT}"                                      # penalized head of the final setting: "pen300"
BASELINE = ("baseline", "model", "single")                         # training, head, reading
REMEDIES = ("student", HEAD, MIXED)

CORE = [(h, r) for h in ("model", HEAD) for r in ("single", MIXED)]
JOBS = {}                                                          # (model, training) -> list of (head, reading)
for _m in MODELS:
    JOBS[(_m, "baseline")] = CORE
    JOBS[(_m, "student")] = (CORE + [(f"pen{g}", MIXED) for g in PENALTY_WEIGHTS if g != PENALTY_WEIGHT]
                             + [(h, r) for h in (HEAD, "model") for r in ("mixed0.25", "mixed0.5", "mixed0.75", "windows")])
for _k in ("windows", "agreement", "selfdistill"):
    JOBS[("ecgfounder", _k)] = [("model", "single")]
print(sum(len(v) for v in JOBS.values()), "configurations from the logits files")

### Evaluation of the configurations

This is the longest cell of the notebook (about 4 min). The results are kept in memory in `RES[(model, training, head, reading)][seed]`.

In [ ]:
RES = {}


def evaluate_jobs(model, training, configs, seeds=SEEDS):
    """Evaluate the (head, reading) combinations `configs` for the fine-tuned models of one kind, one file (seed) at a time."""
    t0 = time.time()
    for seed in seeds:
        tag = model_tag(model, training, seed)
        if not (LOGITS / f"{tag}.npz").exists():
            print(f"   {tag}: no logits file, left out"); continue
        z = load_logits(tag)
        for head in dict.fromkeys(h for h, _ in configs):
            readings = [r for h, r in configs if h == head]
            if f"{head}_single" not in z.files or (any(r != "single" for r in readings) and f"{head}_windows" not in z.files):
                print(f"   {tag}: head {head} is not in the file, left out"); continue
            for reading, p in read_probabilities(z, head, readings).items():
                RES.setdefault((model, training, head, reading), {})[seed] = evaluate(p)
    print(f"{model:10s} {training:11s} {len(configs):2d} configurations  {time.time() - t0:5.0f} s", flush=True)


for (_m, _k), _configs in JOBS.items():
    evaluate_jobs(_m, _k, _configs)

### Ensemble of ten baseline models

The ensemble averages the probabilities of the single reads of ten ECGFounder baseline models trained with different seeds (seeds 0–9, `ENSEMBLE_SEEDS`).
The averaged probability is a predictor of its own, so its thresholds are set for the averaged probabilities, in the same way as for every other predictor.

The ensemble is one predictor, not three. It is compared with each of the three baseline models of the main analysis (seeds 0–2), and the three paired differences are averaged, as for every other configuration.

In [ ]:
_tags = [model_tag("ecgfounder", "baseline", s) for s in ENSEMBLE_SEEDS]
ENSEMBLE = f"ensemble{len(ENSEMBLE_SEEDS)}"
if all((LOGITS / f"{t}.npz").exists() for t in _tags):
    _p = np.mean([read_probabilities(load_logits(t), "model", ["single"])["single"] for t in _tags], 0)      # mean probability of the ten models
    _ev = evaluate(_p)
    RES[("ecgfounder", ENSEMBLE, "model", "single")] = {s: _ev for s in SEEDS}                                # the same predictor, paired with the baseline of each seed
    print(ENSEMBLE, "| decision change between adjacent segments, PTB-XL:", round(100 * float(_ev["unit"]["adjacent_ptbxl"].mean()), 2), "% | AUROC, PTB-XL:", round(_ev["auroc"]["auroc_ptbxl"], 4))
else:
    print("ensemble left out, logits files missing:", [t for t in _tags if not (LOGITS / f"{t}.npz").exists()])

### Monte Carlo dropout

For Monte Carlo dropout the ECGFounder baseline was trained again with dropout (rates 0.1 to 0.5, three seeds each). Notebook 04 stored, for the PTB-XL validation and test rows,
the probability of one read with dropout off (`single`) and the mean probability of 25 passes with dropout on (`mc`). Both are evaluated here as readings of the dropout model:

- reading `mcdropout`: Monte Carlo dropout, the comparator of Fig. 3;
- reading `single`: the same model read once without dropout, which shows how much of the difference comes from training with dropout alone.

Thresholds are set for each of these predictors in the validation fold as usual. Both are compared with the baseline **without dropout** of the same seed.
The comparison is made in PTB-XL only, so only the outcomes of PTB-XL exist for these configurations.

The dropout rate that the paper reports is the one with the highest AUROC in the validation fold with Monte Carlo dropout (mean of the three seeds, the five classes and the two segments). It is selected in the summary below.

In [ ]:
for _rate in DROPOUT_RATES:
    _kind = f"dropout{round(100 * _rate)}"
    for _s in SEEDS:
        _f = MC_DROPOUT / f"{model_tag('ecgfounder', _kind, _s)}.npz"
        if not _f.exists():
            print(f"   {_f.stem}: no file, left out"); continue
        _z = np.load(_f)
        assert set(_z["row"]) == set(VAL) | set(TEST["ptbxl"]), f"{_f.name}: the rows are not the PTB-XL validation and test rows"
        for _reading, _key in (("single", "single"), ("mcdropout", "mc")):
            _p = np.full((N_REC, 2, len(CLASSES)), np.nan, np.float32); _p[_z["row"]] = _z[_key]            # probabilities for the PTB-XL rows only
            RES.setdefault(("ecgfounder", _kind, "model", _reading), {})[_s] = evaluate(_p, ptbxl_only=True)
print(len(RES), "configurations evaluated |", round((time.time() - T_START) / 60, 1), "min since the start")

## Summary tables

### Outcomes of each seed

One row for each configuration and seed: the five thresholds, the AUROC in the validation fold and the value of every outcome. The ensemble is one predictor and has one row (seed −1).

In [ ]:
def config_columns(key):
    model, training, head, reading = key
    return {"model": model, "training": training, "head": head, "reading": reading,
            "penalty_weight": float(head[3:]) if head.startswith("pen") else np.nan,                                  # empty for the model head
            "window_weight": 0.0 if reading == "single" else 1.0 if reading == "windows" else float(reading[5:]) if reading.startswith("mixed") else np.nan,
            "dropout_rate": int(training[7:]) / 100 if training.startswith("dropout") else np.nan}


def seed_row(ev):
    row = {f"threshold_{c}": float(ev["thresholds"][j]) for j, c in enumerate(CLASSES)}
    row["val_auroc"] = ev["val_auroc"]
    for name in OUTCOMES:
        row[name] = 100 * float(ev["unit"][name].mean(dtype=np.float64)) if name in ev["unit"] else ev["auroc"].get(name, np.nan)
    return row


KEYS = sorted(RES, key=lambda k: MODELS.index(k[0]))               # rows in the order of the models; within a model in the order of evaluation
rows = []
for key in KEYS:
    cfg = RES[key]
    one_predictor = len({id(ev) for ev in cfg.values()}) == 1 and len(cfg) > 1        # the ensemble: the same predictor for every seed
    for seed, ev in ([(-1, next(iter(cfg.values())))] if one_predictor else sorted(cfg.items())):
        rows.append({**config_columns(key), "seed": seed, **seed_row(ev)})
BY_SEED = pd.DataFrame(rows)
BY_SEED.to_csv(SUMMARY / "outcomes_by_seed.csv", index=False)
print(BY_SEED.shape)
display(BY_SEED[(BY_SEED["training"] == "baseline") & (BY_SEED["head"] == "model") & (BY_SEED["reading"] == "single")][["model", "seed", "val_auroc", "auroc_ptbxl"] + OUT_ADJACENT + OUT_BETWEEN].round(3))

### Outcomes of each configuration

One row for each configuration. For every outcome the table gives

- the **value**: mean of the three seeds, with the 95% bootstrap CI of the value;
- the **difference** from the baseline of the same foundation model (`baseline`, `model`, `single`): mean paired difference over the three seeds with its 95% CI, and whether the interval excludes zero;
- for decision change, the **relative reduction**: 100 × (1 − value / value of the baseline), from the means of the three seeds. `adjacent_mean4_relative_reduction` is the mean of the relative reductions
  between adjacent segments in the four datasets (Fig. 2c and Fig. 5).

Differences are configuration minus baseline: a negative difference in decision change means fewer decision changes than at baseline, and a negative difference in AUROC or in direction agreement means a loss.

In [ ]:
t0 = time.time()
rows = []
for key in KEYS:
    cfg = RES[key]; ref_key = (key[0], *BASELINE); ref = RES.get(ref_key)
    first = next(iter(cfg.values())); available = set(first["unit"]) | set(first["auroc"])
    row = {**config_columns(key), "n_seeds": len({id(ev) for ev in cfg.values()}), "val_auroc": float(np.mean([ev["val_auroc"] for ev in cfg.values()]))}
    for name in OUTCOMES:
        value = diff = (np.nan, np.nan, np.nan)
        if name in available:
            value = bootstrap_summary(cfg, name)
            if key != ref_key and ref is not None:
                diff = bootstrap_summary(cfg, name, ref)
        row.update({f"{name}_value": value[0], f"{name}_value_lo": value[1], f"{name}_value_hi": value[2], f"{name}_diff": diff[0], f"{name}_diff_lo": diff[1], f"{name}_diff_hi": diff[2],
                    f"{name}_sig": bool(diff[1] > 0 or diff[2] < 0)})                    # False also where no difference is defined (the baseline itself, outcomes that a configuration does not have)
    rows.append(row)
SUM = pd.DataFrame(rows)

# relative reductions from the means of the three seeds
_base = SUM[(SUM["training"] == BASELINE[0]) & (SUM["head"] == BASELINE[1]) & (SUM["reading"] == BASELINE[2])].set_index("model")
for name in OUT_ADJACENT + OUT_BETWEEN:
    SUM[f"{name}_relative_reduction"] = 100 * (1 - SUM[f"{name}_value"] / SUM["model"].map(_base[f"{name}_value"]))
SUM["adjacent_mean4_relative_reduction"] = SUM[[f"{name}_relative_reduction" for name in OUT_ADJACENT]].mean(axis=1, skipna=False)

# dropout rate of the paper: highest validation AUROC with Monte Carlo dropout
_mc = SUM[SUM["reading"] == "mcdropout"]
DROPOUT_SELECTED = float(_mc.loc[_mc["val_auroc"].idxmax(), "dropout_rate"]) if len(_mc) else np.nan
SUM["dropout_selected"] = SUM["dropout_rate"] == DROPOUT_SELECTED
SUM = SUM[list(config_columns(KEYS[0])) + ["dropout_selected"] + [c for c in SUM.columns if c not in config_columns(KEYS[0]) and c != "dropout_selected"]]
SUM.to_csv(SUMMARY / "outcomes_summary.csv", index=False)
print(SUM.shape, "|", round(time.time() - t0), "s")
print("validation AUROC with Monte Carlo dropout by dropout rate:", dict(zip(_mc["dropout_rate"].tolist(), _mc["val_auroc"].round(4).tolist())), "-> selected rate", DROPOUT_SELECTED)

The main rows for each foundation model: baseline (values) and the three remedies (differences from baseline with 95% CIs; an asterisk marks an interval that excludes zero).

In [ ]:
def signed(v, digits):
    """Number with its sign; zero is written without a sign."""
    s = f"{v:+.{digits}f}"
    return s[1:] if float(s) == 0 else s


def interval(point, lo, hi, digits):
    return f"{signed(point, digits)} ({signed(lo, digits)} to {signed(hi, digits)})"


def show(row, names, digits):
    """Value of each outcome for a baseline row, difference from baseline with its CI for any other row."""
    return {n: (f"{row[n + '_value']:.{digits}f}" if np.isnan(row[n + "_diff"]) else interval(row[n + "_diff"], row[n + "_diff_lo"], row[n + "_diff_hi"], digits) + ("*" if row[n + "_sig"] else "")) for n in names}


def pick(model, training, head, reading, table=None):
    """The row of a configuration in the summary table (or in another table with the same first columns)."""
    q = SUM if table is None else table
    return q[(q["model"] == model) & (q["training"] == training) & (q["head"] == head) & (q["reading"] == reading)].iloc[0]


DONE = [m for m in MODELS if (m, *BASELINE) in RES and (m, *REMEDIES) in RES]                 # foundation models with the baseline and the three remedies
_main = [(MODEL_NAMES[m], lab, pick(m, *key)) for m in DONE for lab, key in (("baseline, value", BASELINE), ("three remedies, difference", REMEDIES))]
display(pd.DataFrame([{"model": m, "row": lab, **show(r, OUT_ADJACENT + OUT_BETWEEN + OUT_DIRECTION, 1)} for m, lab, r in _main]))
display(pd.DataFrame([{"model": m, "row": lab, **show(r, ["auroc_ptbxl", "auroc_mimic", "auroc_code15_cd", "auroc_code15_normal", "auroc_echonext"], 3)} for m, lab, r in _main]))

### Direct comparison of the three remedies with the ensemble and with Monte Carlo dropout

Fig. 3 shows each method as a difference from the baseline. The Results also quote the direct comparison: the three remedies **minus** the ensemble, and the three remedies minus Monte Carlo dropout,
for decision change between adjacent segments and for the five-diagnosis AUROC in PTB-XL (ECGFounder). It is the same paired bootstrap with another reference:
the three remedies of seed s are paired with the comparator of seed s (the ensemble is the same predictor for every seed), and the differences are averaged over the three seeds within each resample.
A negative value means fewer decision changes, or a lower AUROC, with the three remedies.

In [ ]:
rows = []
_final = RES.get(("ecgfounder", *REMEDIES))
for key in [k for k in KEYS if _final is not None and (k[1] == ENSEMBLE or k[1].startswith("dropout"))]:
    row = {**config_columns(key), "dropout_selected": config_columns(key)["dropout_rate"] == DROPOUT_SELECTED}
    for name in ("adjacent_ptbxl", "auroc_ptbxl"):
        d, lo, hi = bootstrap_summary(_final, name, RES[key])
        row.update({f"{name}_remedies_minus_this": d, f"{name}_remedies_minus_this_lo": lo, f"{name}_remedies_minus_this_hi": hi, f"{name}_remedies_minus_this_sig": bool(lo > 0 or hi < 0)})
    rows.append(row)
DIRECT = pd.DataFrame(rows)
DIRECT.to_csv(SUMMARY / "direct_comparisons.csv", index=False)
display(DIRECT.drop(columns=["penalty_weight", "window_weight"], errors="ignore").round(4))

### Numbers quoted in the Results

Some numbers of the text are in no table or figure. They follow from the summary tables and are printed here: ranges over the three foundation models,
the relative reduction with each remedy alone and with the remedies applied in part, and the comparison with the ensemble and with Monte Carlo dropout.

In [ ]:
def span(values, digits=1):
    return f"{min(values):.{digits}f}–{max(values):.{digits}f}"


B = {m: pick(m, *BASELINE) for m in DONE}; F = {m: pick(m, *REMEDIES) for m in DONE}
print("Baseline")
print("  AUROC in PTB-XL:", {MODEL_NAMES[m]: round(float(B[m]["auroc_ptbxl_value"]), 3) for m in DONE})
print("  decision change between adjacent segments, % of recordings:", span([B[m][f"{o}_value"] for m in DONE for o in OUT_ADJACENT]),
      "| individual decisions changed, %:", span([B[m][f"individual_{o}_value"] for m in DONE for o in OUT_ADJACENT]))
print("  decision change between recordings, % of pairs: identical reports", span([B[m]["between_same_report_value"] for m in DONE]), "| all pairs", span([B[m]["between_all_value"] for m in DONE]))
print("  MIMIC-IV-ECG, rate between adjacent segments as % of the rate between recordings with identical reports:", span([100 * B[m]["adjacent_mimic_value"] / B[m]["between_same_report_value"] for m in DONE], 0))
print("Three remedies")
print("  decision change between adjacent segments, %:", span([F[m][f"{o}_value"] for m in DONE for o in OUT_ADJACENT]), "| relative reduction, %:", span([F[m][f"{o}_relative_reduction"] for m in DONE for o in OUT_ADJACENT], 0),
      "| by model:", {MODEL_NAMES[m]: span([F[m][f"{o}_relative_reduction"] for o in OUT_ADJACENT], 0) for m in DONE})
print("  reduction in PTB-XL, percentage points:", span([-F[m]["adjacent_ptbxl_diff"] for m in DONE]))
print("  decision change between recordings, % of pairs: identical reports", span([F[m]["between_same_report_value"] for m in DONE]),
      "(relative reduction, %:", span([F[m]["between_same_report_relative_reduction"] for m in DONE], 0) + ") | all pairs", span([F[m]["between_all_value"] for m in DONE]))
print("Relative reduction (%) between adjacent segments as the mean of the four datasets | between recordings with identical reports | share (%) of the reduction with the three remedies, adjacent segments, mean of the four datasets")
for lab, key in (("distillation alone", ("student", "model", "single")), ("mixed reading alone", ("baseline", "model", MIXED)), ("penalized head alone", ("baseline", HEAD, "single")),
                 ("distillation and mixed reading", ("student", "model", MIXED)), ("distillation and penalized head, single read", ("student", HEAD, "single")),
                 ("mixed reading and penalized head, no retraining", ("baseline", HEAD, MIXED)), ("three remedies", REMEDIES)):
    q = {m: pick(m, *key) for m in DONE}
    share = [100 * np.mean([(B[m][f"{o}_value"] - q[m][f"{o}_value"]) / (B[m][f"{o}_value"] - F[m][f"{o}_value"]) for o in OUT_ADJACENT]) for m in DONE]
    print(f"  {lab:48s}", f"{span([q[m]['adjacent_mean4_relative_reduction'] for m in DONE], 0):>6s} | {span([q[m]['between_same_report_relative_reduction'] for m in DONE], 0):>6s} | {span(share, 0) if key != REMEDIES else '100':>6s}")
if len(DIRECT) and len(_mc):
    print(f"ECGFounder, PTB-XL: difference from baseline (dropout rate {DROPOUT_SELECTED:g}); decision change between adjacent segments in percentage points")
    for lab, key in (("ensemble of ten", (ENSEMBLE, "model", "single")), ("Monte Carlo dropout", (f"dropout{round(100 * DROPOUT_SELECTED)}", "model", "mcdropout")), ("three remedies", REMEDIES)):
        r = pick("ecgfounder", *key)
        line = f"  {lab:20s} decision change {show(r, ['adjacent_ptbxl'], 1)['adjacent_ptbxl']}, relative reduction {r['adjacent_ptbxl_relative_reduction']:.0f}%; AUROC {show(r, ['auroc_ptbxl'], 3)['auroc_ptbxl']}"
        if key != REMEDIES:
            d = pick("ecgfounder", *key, table=DIRECT)
            line += " | three remedies minus this: decision change " + interval(*[d["adjacent_ptbxl_remedies_minus_this" + x] for x in ("", "_lo", "_hi")], 1) + "; AUROC " + interval(*[d["auroc_ptbxl_remedies_minus_this" + x] for x in ("", "_lo", "_hi")], 3)
        print(line)

## Files written

Three files in `SUMMARY`. Proportions (decision change, individual decisions, direction agreement) are in **percent** and their differences in **percentage points**. AUROCs and their differences are on the scale 0 to 1.
An empty cell is a missing value, and logical columns contain True or False.
The column `head` has the name of a pandas method: select it with `table["head"]`, not with `table.head`.

### Columns that identify a configuration (in all three files)

| Column | Content |
|---|---|
| `model` | foundation model: `ecgfounder`, `hubert`, `ecgfm` |
| `training` | `baseline`, `student`; for ECGFounder also `windows`, `agreement`, `selfdistill`, `ensemble10`, `dropout10`, `dropout20`, `dropout30`, `dropout40`, `dropout50` |
| `head` | `model` (output layer trained with the network) or `pen{g}` (refitted head with penalty weight g = 0, 30, 100, 300, 1000; `pen0` = refitted without a penalty) |
| `reading` | `single`, `mixed{w}` (w = weight of the window average: `mixed0.25`, `mixed0.5`, `mixed0.75`, `mixed0.9`), `windows` (window average alone), `mcdropout` (Monte Carlo dropout, 25 passes) |
| `penalty_weight` | the number g of the head; empty for the model head |
| `window_weight` | weight of the window average: 0 for `single`, w for `mixed{w}`, 1 for `windows`; empty for `mcdropout` |
| `dropout_rate` | 0.1 … 0.5 for `dropoutNN`; empty otherwise |

The configurations and their names in the paper:

| Name in the paper | `training` | `head` | `reading` | Models |
|---|---|---|---|---|
| Baseline | `baseline` | `model` | `single` | all |
| + Distillation (distillation alone) | `student` | `model` | `single` | all |
| + Mixed reading (distillation and mixed reading) | `student` | `model` | `mixed0.9` | all |
| + Penalized head (three remedies, final setting) | `student` | `pen300` | `mixed0.9` | all |
| Mixed reading alone | `baseline` | `model` | `mixed0.9` | all |
| Penalized head alone | `baseline` | `pen300` | `single` | all |
| Mixed reading and penalized head, no retraining | `baseline` | `pen300` | `mixed0.9` | all |
| Distillation and penalized head, single read | `student` | `pen300` | `single` | all |
| Penalty weight 0 (refit only), 30, 100, 1,000 | `student` | `pen0`, `pen30`, `pen100`, `pen1000` | `mixed0.9` | all |
| Weight of the window average 0, 0.25, 0.5, 0.75, 0.9, 1 with the penalized head | `student` | `pen300` | `single`, `mixed0.25`, `mixed0.5`, `mixed0.75`, `mixed0.9`, `windows` | all |
| The same weights with the model head | `student` | `model` | as above | all |
| Training on 2.5-s windows; direct agreement loss; self-distillation | `windows`; `agreement`; `selfdistill` | `model` | `single` | ECGFounder |
| Ensemble of ten baseline models | `ensemble10` | `model` | `single` | ECGFounder |
| Monte Carlo dropout at a rate of 0.NN | `dropoutNN` | `model` | `mcdropout` | ECGFounder |
| Baseline trained with dropout at a rate of 0.NN, read once with dropout off | `dropoutNN` | `model` | `single` | ECGFounder |

### Names of the outcomes

| Outcome | Definition | Unit resampled |
|---|---|---|
| `adjacent_ptbxl`, `adjacent_mimic`, `adjacent_code15`, `adjacent_echonext` | decision change between adjacent segments: % of recordings in which the decision for at least one of the five classes differs between the two segments | recording |
| `between_same_report`, `between_all` | decision change between recordings: % of pairs in which the decision for at least one class differs between the first segments of the two recordings; pairs with identical machine reports, all pairs (MIMIC-IV-ECG) | pair |
| `individual_adjacent_ptbxl`, `individual_adjacent_mimic`, `individual_adjacent_code15`, `individual_adjacent_echonext`, `individual_between_same_report`, `individual_between_all` | individual decisions changed, %: mean over the five classes of the proportion of decisions that differ between the two readings | recording or pair |
| `adjacent_ptbxl_NORM`, `adjacent_ptbxl_MI`, `adjacent_ptbxl_STTC`, `adjacent_ptbxl_CD`, `adjacent_ptbxl_HYP` | decision change between adjacent segments for one class, PTB-XL, % of recordings | recording |
| `direction_cd`, `direction_normal` | direction agreement in CODE-15%, % of pairs: conduction disturbance, normal ECG | pair |
| `auroc_ptbxl`, `auroc_mimic` | mean AUROC of the five classes, PTB-XL test fold, MIMIC-IV-ECG test set | recording |
| `auroc_ptbxl_{CLASS}`, `auroc_mimic_{CLASS}` | AUROC of one class (`NORM`, `MI`, `STTC`, `CD`, `HYP`) | recording |
| `auroc_code15_cd`, `auroc_code15_normal` | AUROC for conduction disturbance and for normal ECG, CODE-15% | recording |
| `auroc_echonext` | AUROC of the hypertrophy output against a wall thickness of 13 mm or more, EchoNext | recording |

The configurations of `ensemble10` have all outcomes. Those of `dropoutNN` have the outcomes of PTB-XL only (`adjacent_ptbxl`, `individual_adjacent_ptbxl`, `adjacent_ptbxl_{CLASS}`, `auroc_ptbxl`, `auroc_ptbxl_{CLASS}`); their other outcomes are empty.

### `outcomes_summary.csv`

**One row = one configuration** (mean of the three seeds). This is the file for the tables and figures.

| Column | Content |
|---|---|
| `model`, `training`, `head`, `reading`, `penalty_weight`, `window_weight`, `dropout_rate` | the configuration, as above |
| `dropout_selected` | True for the rows of the dropout rate with the highest validation AUROC with Monte Carlo dropout (both readings of that rate); False otherwise. Monte Carlo dropout of Fig. 3 = the row with `reading` = `mcdropout` and `dropout_selected` = True |
| `n_seeds` | number of fine-tuned models averaged (3; 1 for the ensemble, which is one predictor compared with the baseline of each of the three seeds) |
| `val_auroc` | AUROC in the PTB-XL validation fold with this head and reading: mean of the five classes, the two segments and the seeds |
| `{outcome}_value` | value of the outcome, mean of the three seeds |
| `{outcome}_value_lo`, `{outcome}_value_hi` | 95% bootstrap CI of the value |
| `{outcome}_diff` | paired difference from the baseline of the same foundation model (configuration minus baseline), mean of the three seeds; percentage points for proportions. Empty in the baseline row |
| `{outcome}_diff_lo`, `{outcome}_diff_hi` | 95% bootstrap CI of the difference |
| `{outcome}_sig` | True if the CI of the difference excludes zero; False otherwise, also where no difference is defined |
| `{outcome}_relative_reduction` | only for `adjacent_ptbxl`, `adjacent_mimic`, `adjacent_code15`, `adjacent_echonext`, `between_same_report` and `between_all`: relative reduction in %, 100 × (1 − value / value of the baseline), from the means of the three seeds; 0 in the baseline row |
| `adjacent_mean4_relative_reduction` | mean of the four `adjacent_{dataset}_relative_reduction` (empty for `dropoutNN`, which has PTB-XL only) |

`{outcome}` is every name of the table of outcomes above (34 outcomes, seven columns each). The relative reductions are the last columns of the file, and the last cell of this notebook prints all column names.
Example: the difference of Table 2 for decision change in PTB-XL with ECGFounder is `adjacent_ptbxl_diff` (with `adjacent_ptbxl_diff_lo` and `adjacent_ptbxl_diff_hi`) in the row
`model` = `ecgfounder`, `training` = `student`, `head` = `pen300`, `reading` = `mixed0.9`, and the baseline value is `adjacent_ptbxl_value` in the row `ecgfounder`, `baseline`, `model`, `single`.

### `outcomes_by_seed.csv`

**One row = one configuration and one seed** (one predictor). The ensemble has one row with `seed` = −1.

| Column | Content |
|---|---|
| `model`, `training`, `head`, `reading`, `penalty_weight`, `window_weight`, `dropout_rate` | the configuration, as above |
| `seed` | training seed of the model (0, 1, 2); −1 for the ensemble |
| `threshold_NORM`, `threshold_MI`, `threshold_STTC`, `threshold_CD`, `threshold_HYP` | decision thresholds of this predictor (probability; 90% sensitivity in the validation fold) |
| `val_auroc` | AUROC in the validation fold, mean of the five classes and the two segments |
| `{outcome}` | value of the outcome for this predictor (no suffix) |

### `direct_comparisons.csv`

**One row = one comparator** of Fig. 3 (ECGFounder, PTB-XL): the ensemble, and each dropout rate with both readings.

| Column | Content |
|---|---|
| `model`, `training`, `head`, `reading`, `penalty_weight`, `window_weight`, `dropout_rate` | the configuration of the comparator |
| `dropout_selected` | as in `outcomes_summary.csv` |
| `adjacent_ptbxl_remedies_minus_this` | decision change between adjacent segments in PTB-XL: three remedies minus the comparator, in percentage points, mean of the three seeds (negative = fewer decision changes with the three remedies) |
| `adjacent_ptbxl_remedies_minus_this_lo`, `adjacent_ptbxl_remedies_minus_this_hi` | 95% bootstrap CI of this difference |
| `adjacent_ptbxl_remedies_minus_this_sig` | True if the CI excludes zero |
| `auroc_ptbxl_remedies_minus_this`, `auroc_ptbxl_remedies_minus_this_lo`, `auroc_ptbxl_remedies_minus_this_hi`, `auroc_ptbxl_remedies_minus_this_sig` | the same for the five-diagnosis AUROC in PTB-XL (negative = lower AUROC with the three remedies) |

The files as written, with all their columns:

In [ ]:
import textwrap

for _f in ("outcomes_summary.csv", "outcomes_by_seed.csv", "direct_comparisons.csv"):
    _t = pd.read_csv(SUMMARY / _f)
    print(f"{_f}: {_t.shape[0]} rows, {_t.shape[1]} columns")
    print(textwrap.fill(", ".join(_t.columns), width=220, initial_indent="    ", subsequent_indent="    "), "\n")
print(f"total time {(time.time() - T_START) / 60:.1f} min")